# Week 39, Day 5: Phase 3 exam

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload or close the page. It doesn't stop you working when time is up: that's on your honour..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: B1: Intervals

Write `Interval(lo, hi)`, the whole numbers from `lo` up to but not including `hi` (`lo > hi` raises `ValueError`; `lo == hi` is an empty interval). Give it `__repr__` (`Interval(2, 5)`), `__len__`, `__contains__` (for whole numbers), `__eq__` and `__hash__`, `__iter__` (the numbers, in order), and `__and__`: `a & b` is their overlap as an `Interval`, or `Interval(0, 0)` if they don't overlap. All empty intervals are equal to each other.

In [ ]:
class Interval:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"basics\"\n    a = Interval(2, 5)\n    assert repr(a) == \"Interval(2, 5)\" and len(a) == 3 and 2 in a and 5 not in a and list(a) == [2, 3, 4]\n    try:\n        Interval(5, 2)\n        assert False\n    except ValueError:\n        pass\ndef test_overlap():\n    \"overlaps\"\n    assert Interval(2, 5) & Interval(4, 9) == Interval(4, 5) and Interval(0, 3) & Interval(3, 6) == Interval(0, 0)\n    assert Interval(0, 10) & Interval(2, 4) == Interval(2, 4) and len(Interval(1, 2) & Interval(5, 7)) == 0\n    assert Interval(3, 3) == Interval(7, 7) and len({Interval(3, 3), Interval(0, 0), Interval(1, 2)}) == 2\n    assert (Interval(1, 2) == (1, 2)) is False")

### Exercise: B2: Money in currencies

Write `CurrencyError(ValueError)` and `Money(pence, currency)`, with the currency stored in upper case. It supports `+` and `-` with Money in the **same** currency (another currency raises `CurrencyError("can't mix GBP and NGN")`, in the order of the operands), comparison with `<`, `<=`, `>` and `>=` (same rule; use `functools.total_ordering`), `==` (different currencies are simply not equal) and `__hash__`, and `str(m)` like `GBP 12.50` or `GBP -0.05`.

In [ ]:
from functools import total_ordering

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_money():\n    \"money\"\n    a, b = Money(1250, \"gbp\"), Money(75, \"GBP\")\n    assert str(a + b) == \"GBP 13.25\" and str(b - a) == \"GBP -11.75\" and str(Money(-5, \"GBP\")) == \"GBP -0.05\"\n    assert b < a and a >= b and Money(1, \"gbp\") == Money(1, \"GBP\") and Money(1, \"GBP\") != Money(1, \"NGN\")\n    assert len({Money(1, \"GBP\"), Money(1, \"gbp\"), Money(1, \"NGN\")}) == 2\ndef test_currency():\n    \"mixing currencies\"\n    for call in [lambda: Money(1, \"GBP\") + Money(1, \"NGN\"), lambda: Money(1, \"GBP\") < Money(1, \"NGN\")]:\n        try:\n            call()\n            assert False\n        except CurrencyError as err:\n            assert str(err) == \"can't mix GBP and NGN\" and isinstance(err, ValueError)")

### Exercise: B3: Predict

Answer each without running code.

In [ ]:
from dataclasses import dataclass, field

class Base:
    count = 0
    def __init__(self):
        Base.count += 1
        self.id = Base.count

class Child(Base):
    def __init__(self, name):
        super().__init__()
        self.name = name

@dataclass(order=True)
class Job:
    priority: int
    name: str = field(compare=False)

kids = [Child("a"), Child("b")]
jobs = sorted([Job(2, "x"), Job(1, "z"), Job(2, "a")])

a = ___   # [k.id for k in kids]
b = ___   # Base.count
c = ___   # [j.name for j in jobs]
d = ___   # Job(1, "p") == Job(1, "q")
e = ___   # [c.__name__ for c in Child.__mro__]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 2] and b == 2 and c == [\"z\", \"x\", \"a\"] and d is True and e == [\"Child\", \"Base\", \"object\"]")

### Exercise: B4: An order's life

Write `BadMove(ValueError)` and `Order`, a state machine with a transition table `MOVES` from `(state, event)` to the new state: placed + pay → paid; placed + cancel → cancelled; paid + ship → shipped; paid + cancel → refunded; shipped + deliver → delivered. An order starts `"placed"`. `handle(event)` moves it and returns the new state, or raises `BadMove("can't EVENT a STATE order")`. `undo()` returns the order to its previous state (any number of times back to `"placed"`) and returns `True`, or `False` when there's nothing to undo. `history` is a list of the states it has been in, oldest first, **not** including the current one.

In [ ]:
class Order:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_moves():\n    \"moves\"\n    o = Order()\n    assert o.state == \"placed\" and [o.handle(e) for e in [\"pay\", \"ship\", \"deliver\"]] == [\"paid\", \"shipped\", \"delivered\"]\n    assert o.history == [\"placed\", \"paid\", \"shipped\"]\n    for start, event in [(\"delivered\", \"cancel\")]:\n        try:\n            o.handle(event)\n            assert False\n        except BadMove as err:\n            assert str(err) == \"can't cancel a delivered order\" and isinstance(err, ValueError)\n    assert Order().handle(\"cancel\") == \"cancelled\"\n    p = Order()\n    p.handle(\"pay\")\n    assert p.handle(\"cancel\") == \"refunded\"\ndef test_undo():\n    \"undo\"\n    o = Order()\n    assert o.undo() is False\n    o.handle(\"pay\")\n    o.handle(\"ship\")\n    assert o.undo() and o.state == \"paid\" and o.history == [\"placed\"]\n    assert o.undo() and o.state == \"placed\" and o.undo() is False\n    assert len(Order.MOVES) == 5")

### Exercise: B5: A feed

Write `Feed`, which holds posts (text) and tells observers about new ones. `post(text)` adds a post (blank text raises `ValueError`) and calls each subscriber with the text; `subscribe(listener)` and `unsubscribe(listener)`. A `Feed` is **iterable**, giving the posts **newest first**, and can be looped over any number of times; a loop that is under way isn't affected by posts added during it (it works on the posts at the moment the loop started). `len(feed)` is the number of posts.

In [ ]:
class Feed:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_feed():\n    \"posts and loops\"\n    f = Feed()\n    seen = []\n    f.subscribe(seen.append)\n    for t in [\"a\", \"b\", \"c\"]:\n        f.post(t)\n    assert list(f) == [\"c\", \"b\", \"a\"] and list(f) == [\"c\", \"b\", \"a\"] and len(f) == 3 and seen == [\"a\", \"b\", \"c\"]\n    try:\n        f.post(\"  \")\n        assert False\n    except ValueError:\n        pass\n    f.unsubscribe(seen.append)\n    f.post(\"d\")\n    assert seen == [\"a\", \"b\", \"c\"]\ndef test_snapshot():\n    \"loops see a snapshot\"\n    f = Feed()\n    f.post(\"x\")\n    f.post(\"y\")\n    got = []\n    for text in f:\n        got.append(text)\n        if text == \"y\":\n            f.post(\"z\")\n    assert got == [\"y\", \"x\"] and list(f) == [\"z\", \"y\", \"x\"]")

### Exercise: B6: Typed helpers

Write two generic functions with full type hints (use the `def name[T](...)` form):

- `group_by[T, K](items: list[T], key: Callable[[T], K]) -> dict[K, list[T]]`: the items grouped by their key, keys in the order first seen, items in their original order;
- `first_where[T](items: list[T], test: Callable[[T], bool]) -> T | None`: the first item passing the test, or `None`.

In [ ]:
from typing import Callable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import typing\ndef test_values():\n    \"behaviour\"\n    assert group_by([\"apple\", \"fig\", \"avocado\", \"kiwi\"], lambda w: w[0]) == {\"a\": [\"apple\", \"avocado\"], \"f\": [\"fig\"], \"k\": [\"kiwi\"]}\n    assert list(group_by([3, 1, 2], lambda n: n % 2)) == [1, 0] and group_by([], len) == {}\n    assert first_where([1, 4, 6], lambda n: n % 2 == 0) == 4 and first_where([1, 3], lambda n: n > 5) is None\ndef test_hints():\n    \"type hints\"\n    T, K = group_by.__type_params__\n    a = group_by.__annotations__\n    assert a[\"items\"] == list[T] and a[\"key\"] == typing.Callable[[T], K] and a[\"return\"] == dict[K, list[T]]\n    (U,) = first_where.__type_params__\n    b = first_where.__annotations__\n    assert b[\"items\"] == list[U] and b[\"test\"] == typing.Callable[[U], bool] and b[\"return\"] == (U | None)")

### Bug hunt: B7: Three bugs

`Team` and `Squad` have three bugs: every team shares one list of members, teams can't be put in a set even though they define equality, and a `Squad` has no name. Fix them, without changing what each method is for.

In [ ]:
class Team:
    members = []

    def __init__(self, name):
        self.name = name

    def add(self, person):
        if person in self.members:
            raise ValueError(f"{person} is already in {self.name}")
        self.members.append(person)

    def __eq__(self, other):
        if not isinstance(other, Team):
            return NotImplemented
        return self.name == other.name

class Squad(Team):
    def __init__(self, name, coach):
        self.coach = coach

    def describe(self):
        return f"{self.name} (coach {self.coach}): {', '.join(self.members)}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"fixed\"\n    a, b = Team(\"Lions\"), Team(\"Tigers\")\n    a.add(\"Ada\")\n    assert b.members == [] and a.members == [\"Ada\"]\n    b.add(\"Ada\")\n    assert len({Team(\"Lions\"), Team(\"Lions\"), b}) == 2\n    s = Squad(\"Hawks\", \"Bo\")\n    s.add(\"Chi\")\n    assert s.describe() == \"Hawks (coach Bo): Chi\" and s == Team(\"Hawks\")\n    try:\n        s.add(\"Chi\")\n        assert False\n    except ValueError as err:\n        assert str(err) == \"Chi is already in Hawks\"")

### Exercise: B8: Test a password policy

`check(password)` in `policy.py` returns a list of the problems with a password (an empty list if it's fine): `"too short"` (fewer than 10 characters), `"no digit"`, `"no capital"`, and `"contains spaces"`, in that order. Write pytest tests that catch all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('policy.py', 'w', encoding='utf-8') as f:
    f.write('def check(password):\n    problems = []\n    if len(password) < 10:\n        problems.append("too short")\n    if not any(c.isdigit() for c in password):\n        problems.append("no digit")\n    if not any(c.isupper() for c in password):\n        problems.append("no capital")\n    if " " in password:\n        problems.append("contains spaces")\n    return problems\n')

In [ ]:
import pytest
from policy import check

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"policy.py\").read()\nBROKEN = {\n    \"ten is too short\": GOOD.replace(\"if len(password) < 10:\", \"if len(password) <= 10:\"),\n    \"nine is long enough\": GOOD.replace(\"if len(password) < 10:\", \"if len(password) < 9:\"),\n    \"digit check inverted\": GOOD.replace(\"if not any(c.isdigit() for c in password):\", \"if any(c.isdigit() for c in password):\"),\n    \"lower case counts as a capital\": GOOD.replace(\"c.isupper()\", \"c.isalpha()\"),\n    \"spaces allowed\": GOOD.replace('    if \" \" in password:\\n        problems.append(\"contains spaces\")\\n', \"\"),\n    \"only the first problem\": GOOD.replace(\"    return problems\", \"    return problems[:1]\"),\n}\ndef test_catches():\n    \"your tests pass on the real code and catch all six broken versions\"\n    results = against(source, \"policy\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"policy.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: C: A league table

A five-a-side league records results typed by a volunteer, like `Red Lions 3-1 Blue Tigers`. Build it in layers.

**Errors.** `LeagueError(Exception)` is the base; `UnknownTeam(LeagueError, LookupError)`; `BadResult(LeagueError, ValueError)`.

**Domain.** `Result`, a frozen dataclass with `home`, `away`, `home_goals` and `away_goals`, and `__str__` giving `Red Lions 3-1 Blue Tigers`. `parse_result(line)` reads that form: the score is the one word made of digits, a `-`, and digits; the home team is the words before it and the away team the words after (both stripped, with single spaces). It raises `BadResult(f"bad result: {line!r}")` if there's no such word, either team is missing, or the two teams are the same.

**Points are a strategy**: a function `(goals_for, goals_against) -> points`. `standard_points` gives 3 for a win, 1 for a draw and 0 for a loss.

**Service.** `League(teams, points=standard_points)`, where `teams` is a list of names:

- `record(line)` parses the line, checks both teams are in the league (otherwise `UnknownTeam(f"no team {name!r}")`, home team checked first), stores the result and returns it;
- `undo()` removes and returns the most recent result, or raises `LeagueError("nothing to undo")`;
- `table()` returns one tuple per team, `(team, played, won, drawn, lost, goals_for, goals_against, points)`, sorted by points (most first), then goal difference (most first), then goals for (most first), then name (A to Z). Every team in the league appears, even with no games.

**Interface.** `handle(line, league)`: `table` gives one line per team, `1. Red Lions P2 W1 D1 L0 F4 A2 Pts4` (the position, then the figures); `undo` gives `removed Red Lions 3-1 Blue Tigers`; anything else is a result, giving `recorded Red Lions 3-1 Blue Tigers`. A `LeagueError` becomes `error: ` and its message.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TEAMS = [\"Red Lions\", \"Blue Tigers\", \"Owls\", \"Green Bees\"]\ndef test_parse():\n    \"reading results\"\n    r = parse_result(\"  Red   Lions 3-1 Blue Tigers \")\n    assert r == Result(\"Red Lions\", \"Blue Tigers\", 3, 1) and str(r) == \"Red Lions 3-1 Blue Tigers\"\n    assert parse_result(\"Owls 10-0 Green Bees\").home_goals == 10\n    for line in [\"Owls 3 1 Bees\", \"3-1 Owls\", \"Owls 3-1\", \"Owls 3-1 Owls\", \"Owls x-1 Bees\", \"\"]:\n        try:\n            parse_result(line)\n            assert False, f\"{line!r} should raise BadResult\"\n        except BadResult as err:\n            assert str(err) == f\"bad result: {line!r}\" and isinstance(err, ValueError)\ndef test_points():\n    \"the points strategy\"\n    assert [standard_points(2, 1), standard_points(1, 1), standard_points(0, 3)] == [3, 1, 0]\ndef test_table():\n    \"the table\"\n    lg = League(TEAMS)\n    for line in [\"Red Lions 3-1 Blue Tigers\", \"Owls 2-2 Red Lions\", \"Blue Tigers 1-0 Owls\"]:\n        lg.record(line)\n    assert lg.table() == [(\"Red Lions\", 2, 1, 1, 0, 5, 3, 4), (\"Blue Tigers\", 2, 1, 0, 1, 2, 3, 3), (\"Owls\", 2, 0, 1, 1, 2, 3, 1), (\"Green Bees\", 0, 0, 0, 0, 0, 0, 0)]\n    assert str(lg.undo()) == \"Blue Tigers 1-0 Owls\" and lg.table()[1][0] == \"Owls\"\ndef test_ties():\n    \"tie-breaks\"\n    lg = League([\"D\", \"C\", \"B\", \"A\"])\n    lg.record(\"A 1-0 B\")\n    lg.record(\"C 3-2 D\")\n    lg.record(\"B 0-0 D\")\n    assert [row[0] for row in lg.table()] == [\"C\", \"A\", \"D\", \"B\"]\n    lg2 = League([\"Zed\", \"Amy\"])\n    assert [row[0] for row in lg2.table()] == [\"Amy\", \"Zed\"]\ndef test_errors():\n    \"errors\"\n    lg = League(TEAMS)\n    for call, error, message in [(lambda: lg.record(\"Wolves 1-0 Hares\"), UnknownTeam, \"no team 'Wolves'\"), (lambda: lg.record(\"Owls 1-0 Hares\"), UnknownTeam, \"no team 'Hares'\"),\n                                 (lg.undo, LeagueError, \"nothing to undo\")]:\n        try:\n            call()\n            assert False, message\n        except error as err:\n            assert str(err) == message\n    assert issubclass(UnknownTeam, LookupError) and issubclass(BadResult, LeagueError)\ndef test_strategy():\n    \"another points rule\"\n    lg = League([\"A\", \"B\"], points=lambda f, a: 2 if f > a else (1 if f == a else 0))\n    lg.record(\"A 2-0 B\")\n    assert lg.table()[0] == (\"A\", 1, 1, 0, 0, 2, 0, 2)\ndef test_handle():\n    \"the interface\"\n    lg = League(TEAMS)\n    assert handle(\"Red Lions 3-1 Blue Tigers\", lg) == \"recorded Red Lions 3-1 Blue Tigers\"\n    assert handle(\"Owls 2-2 Red Lions\", lg) == \"recorded Owls 2-2 Red Lions\"\n    assert handle(\"table\", lg).splitlines()[0] == \"1. Red Lions P2 W1 D1 L0 F5 A3 Pts4\"\n    assert handle(\"table\", lg).splitlines()[2:] == [\"3. Green Bees P0 W0 D0 L0 F0 A0 Pts0\", \"4. Blue Tigers P1 W0 D0 L1 F1 A3 Pts0\"]\n    assert handle(\"undo\", lg) == \"removed Owls 2-2 Red Lions\"\n    assert handle(\"Owls 1-1 Owls\", lg) == \"error: bad result: 'Owls 1-1 Owls'\" and handle(\"Bats 1-0 Owls\", lg) == \"error: no team 'Bats'\"")

## Before you start

**The rules.** You have **2 hours and 50 minutes** in total. Work alone. Close your notes, don't use the tutor, and don't search the web: the point is to find out what **you** can do, and the debrief can only help with gaps it can see. You may run your code and use **Check my answer** as often as you like.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 20 quiz questions | 20 (1 each) |
| B | 8 problems | 110 |
| C | one larger program | 50 |
| | **Total** | **180** |

Count a problem's full marks only when every check passes. **108 (60%)** passes Phase 3; **144 (80%)** is a distinction. If you don't pass, tomorrow's debrief tells you what to revise, and you can take the exam again after the rest week.

**Strategy.** Read every problem first. Do the ones you're sure of first, and don't stay stuck on one for more than 20 minutes. Test small and awkward cases before pressing Check.

## Part A: quiz

## Part B: eight problems

## Part C: a larger program

## When you finish

Add up your marks: 1 for each quiz question you got right first time, and the full marks for each problem whose checks all pass. Write your total down, with the time you took, and a note of every problem you didn't finish or found hard. Don't look anything up yet.

Tomorrow's debrief goes through every question and problem: a model solution, why it works, the mistakes people most often make, and what to revise for each one.